# 1、 HumanInTheLoopMiddleware中间件
## 1.1举例过程1：工程调用的中断

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    }
)

In [ ]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint
from langchain_deepseek import ChatDeepSeek  # 或使用 ChatOpenAI


# ===== 1. 定义工具 =====
@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """查询指定城市天气
    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res  # 修正：原为 return re，改为 return res


@tool
def get_news() -> str:
    """查询当日新闻"""
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


# ===== 2. 初始化模型（根据实际情况选择）=====
# 使用 DeepSeek（需安装 langchain-deepseek）
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=os.getenv("DEEPSEEK_API_KEY", "your-api-key-here"),
    temperature=0.7,
)
# 如果使用 OpenAI，可以替换为：
# from langchain_openai import ChatOpenAI
# model = ChatOpenAI(model="gpt-4o", temperature=0.7)


# ===== 3. 创建 Agent =====
agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断啦",
                },
            },
            description_prefix="中断啦",
        ),
    ],
)

config = {"configurable": {"thread_id": "1"}}

# ===== 4. 第一次调用（会因发送邮件而中断）=====
user_request = (
    "请帮我查询今天北京的天气，"
    "查询今日新闻，"
    "查看ID为 'sk2131421' 的邮件内容，"
    "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'，"
    "同时做这四件事。"
)
response = agent.invoke(
    {"messages": [HumanMessage(content=user_request)]},
    config=config,
)

print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)
print("========= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()

# ===== 5. 查看中断信息 =====
interrupts = response.get("__interrupt__", [])
print("========== interrupts ==========")
rprint(interrupts)

# 提取 action_requests（注意 LangGraph 中的访问路径）
if interrupts:
    # 通常 interrupts[0].value 包含 action_requests
    # 这里需要根据实际结构调整，示例写法：
    for interrupt in interrupts:
        value = interrupt.value
        if isinstance(value, dict) and "action_requests" in value:
            action_requests = value["action_requests"]
            for action_request in action_requests:
                rprint(action_request)
        else:
            # 直接打印整个中断信息
            rprint(value)
else:
    print("没有中断信息，Agent 已完成所有操作。")

## 举例过程2：指明工具调用请求决策

In [ ]:


weather_decision = {
    "type":"edit",
    "edited_action":{
    "name":"get_weather",
    "args":{"city":"上海市","is_forcast":True},
                    }
}
news_decision = {
    "type":"approve"
}
send_email_decision = {
    "type":"approve"
}




decisions = {
    "decisions": [
        {}
    ]
}

interrupts = response.get("__interrupt__",[])
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    if action_request["action"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["action"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["action"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)
if interrupts:
    resumed_response = agent.invoke(
    Command(resume=decisions),
    config=config

)
    for msg in resumed_response["messages"]:
        msg.pretty_print()